# Coupon Received 로그 SQL 생성

## 쿠폰 목록 (8종)
- 연중 발급 가능: WELCOME5000, COMEBACK20, REGULAR5000, FIRSTORDER10
- 시즌 한정 발급:
  - SPRING20 → 3,4,5월
  - SUMMER10000 → 6,7,8월
  - AUTUMN20 → 9,10,11월
  - WINTER10000 → 12,1,2월

## 규칙
- 유저당 같은 쿠폰 중복 수령 불가 (유저 100명 x 쿠폰 8종 = 최대 800건)
- TARGET_COUNT(약 500건)에 맞춰 (user_id, coupon) 조합을 랜덤 샘플링
- 시즌 쿠폰은 해당 시즌 월(月) 내에서만 event_timestamp 생성
- discountAmount: RATE 타입은 0 (FE 버그 재현), FIXED 타입은 실제 할인 금액
- expiryDate: event_timestamp 날짜 + expired_days (쿠폰별 7일 또는 30일)
- user_id: 1~100 (로그인 사용자만)
- user_login_id: user0001~user0100
- client_uuid: 세션마다 고유 UUID
- 생성된 매핑은 coupon_received_map.json으로 저장 (coupon_used 생성 시 참조)

In [1]:
import random
import json
import uuid
from datetime import datetime, timedelta

In [2]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
START_DATE = datetime(2025, 6, 1, 0, 0, 0)
END_DATE   = datetime(2026, 6, 16, 23, 59, 59)

# 유저 수
USER_COUNT = 100

# 목표 발급 건수 (유저당 같은 쿠폰 중복 수령 불가 -> 최대 USER_COUNT x len(COUPONS))
TARGET_COUNT = 500

# 쿠폰 목록
# season: None이면 연중 발급 가능, 리스트면 해당 월에만 발급 가능
COUPONS = [
    {'code': 'WELCOME5000',  'discount_amount': 5000, 'discount_type': 'FIXED', 'expired_days': 30, 'season': None},
    {'code': 'COMEBACK20',   'discount_amount': 20,   'discount_type': 'RATE',  'expired_days': 7,  'season': None},
    {'code': 'REGULAR5000',  'discount_amount': 5000, 'discount_type': 'FIXED', 'expired_days': 7,  'season': None},
    {'code': 'FIRSTORDER10', 'discount_amount': 10,   'discount_type': 'RATE',  'expired_days': 7,  'season': None},
    {'code': 'SPRING20',     'discount_amount': 20,   'discount_type': 'RATE',  'expired_days': 7,  'season': [3, 4, 5]},
    {'code': 'SUMMER10000',  'discount_amount': 10000,'discount_type': 'FIXED', 'expired_days': 7,  'season': [6, 7, 8]},
    {'code': 'AUTUMN20',     'discount_amount': 20,   'discount_type': 'RATE',  'expired_days': 7,  'season': [9, 10, 11]},
    {'code': 'WINTER10000',  'discount_amount': 10000,'discount_type': 'FIXED', 'expired_days': 7,  'season': [12, 1, 2]},
]

In [3]:
def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

def random_datetime_in_months(start, end, months):
    """start~end 범위 내에서, 월(month)이 months 리스트에 포함되는 날짜만 골라 랜덤 datetime 반환"""
    candidates = [m for m in range(start.year * 12 + (start.month - 1), end.year * 12 + (end.month - 1) + 1)]
    valid_year_months = []
    for ym in candidates:
        y, m = divmod(ym, 12)
        m += 1
        if m in months:
            valid_year_months.append((y, m))

    while True:
        y, m = random.choice(valid_year_months)
        # 해당 월의 일/시/분/초 랜덤 (최대 28일로 단순화하여 월 범위 초과 방지)
        day = random.randint(1, 28)
        hour = random.randint(0, 23)
        minute = random.randint(0, 59)
        second = random.randint(0, 59)
        candidate = datetime(y, m, day, hour, minute, second)
        if start <= candidate <= end:
            return candidate

def format_kst(dt):
    """event_timestamp 포맷 (KST +09:00)"""
    return dt.strftime('%Y-%m-%dT%H:%M:%S.') + f"{dt.microsecond // 1000:03d}+09:00"

def format_history_ts(dt):
    """history_timestamp 포맷 (마이크로초 포함)"""
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')

def format_date(dt):
    """expiryDate 포맷 (YYYY-MM-DD)"""
    return dt.strftime('%Y-%m-%d')

In [4]:
# 유저(1~100) x 쿠폰(8종) 전체 조합 생성 (유저당 같은 쿠폰 중복 수령 불가 -> 최대 800)
all_combinations = [
    (user_id, coupon)
    for user_id in range(1, USER_COUNT + 1)
    for coupon in COUPONS
]

# TARGET_COUNT 만큼 랜덤 샘플링 (중복 없음)
combinations = random.sample(all_combinations, min(TARGET_COUNT, len(all_combinations)))
random.shuffle(combinations)

rows = []
received_map = []  # coupon_used 생성용

for user_id, coupon in combinations:
    user_login_id = f'user{user_id:04d}'

    coupon_code   = coupon['code']
    discount_type = coupon['discount_type']
    expired_days  = coupon['expired_days']
    season        = coupon['season']

    # 시즌 쿠폰은 해당 시즌 월에서만 event_timestamp 생성, 연중 쿠폰은 전체 기간에서 랜덤
    if season is not None:
        event_ts = random_datetime_in_months(START_DATE, END_DATE, season)
    else:
        event_ts = random_datetime(START_DATE, END_DATE)

    history_ts = event_ts + timedelta(seconds=1)

    # RATE 타입은 FE 버그 재현 -> discount_amount = 0
    # FIXED 타입은 실제 금액
    if discount_type == 'RATE':
        discount_amount = 0
    else:
        discount_amount = coupon['discount_amount']

    # expiryDate = 발급일 + expired_days
    expiry_date = format_date(event_ts + timedelta(days=expired_days))

    client_uuid = str(uuid.uuid4())

    json_log = json.dumps({
        'event_name':      'coupon_received',
        'couponCode':      coupon_code,
        'discountAmount':  discount_amount,
        'expiryDate':      expiry_date,
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(event_ts)
    }, ensure_ascii=False)

    rows.append((history_ts, json_log))

    received_map.append({
        'user_id':           user_id,
        'user_login_id':     user_login_id,
        'coupon_code':       coupon_code,
        'discount_amount':   discount_amount,
        'expired_days':      expired_days,
        'received_event_ts': format_kst(event_ts),
    })

print(f'✅ {len(rows)}개 coupon_received 로그 생성 완료 (유저 {USER_COUNT}명 x 쿠폰 {len(COUPONS)}종, 유저당 쿠폰 중복 없음)')

✅ 500개 coupon_received 로그 생성 완료 (유저 100명 x 쿠폰 8종, 유저당 쿠폰 중복 없음)


In [5]:
# SQL 생성 및 저장
lines  = ['INSERT INTO first_save_history (history_timestamp, json_log) VALUES']
values = []

for history_ts, json_log in rows:
    ts_str  = format_history_ts(history_ts)
    escaped = json_log.replace("'", "''")
    values.append(f"  ('{ts_str}', '{escaped}')")

lines.append(',\n'.join(values) + ';')
sql = '\n'.join(lines)

with open('coupon_received_logs.sql', 'w', encoding='utf-8') as f:
    f.write(sql)

print(f'✅ {len(rows)}개 coupon_received 로그 SQL 생성 완료 → coupon_received_logs.sql')

✅ 500개 coupon_received 로그 SQL 생성 완료 → coupon_received_logs.sql


In [6]:
# coupon_used 생성 시 참조할 매핑 정보 저장
with open('coupon_received_map.json', 'w', encoding='utf-8') as f:
    json.dump(received_map, f, ensure_ascii=False, indent=2)

print(f'✅ coupon_received_map.json 저장 완료 ({len(received_map)}건)')

✅ coupon_received_map.json 저장 완료 (500건)


In [7]:
# ── 미리보기 ──
print('=== COUPON RECEIVED SQL (앞 500자) ===')
print(sql[:500])

=== COUPON RECEIVED SQL (앞 500자) ===
INSERT INTO first_save_history (history_timestamp, json_log) VALUES
  ('2025-10-17 22:08:37.000000', '{"event_name": "coupon_received", "couponCode": "AUTUMN20", "discountAmount": 0, "expiryDate": "2025-10-24", "user_id": 31, "user_login_id": "user0031", "client_uuid": "45b24112-bb22-4b58-968a-09c21697eb64", "event_timestamp": "2025-10-17T22:08:36.000+09:00"}'),
  ('2025-07-11 06:33:26.000000', '{"event_name": "coupon_received", "couponCode": "REGULAR5000", "discountAmount": 5000, "expiryDate": 
